# Go2 handstand curriculum V2
Use Python 3.11. Upload the provided `handstand_v2_molab*.zip` bundle alongside this notebook.
All checkpoint loads are from your own trusted archive. Existing output directories are never reused.
Run diagnostics and tests before PPO. Extended training is a separate authorization and requires replicated supported improvement.

In [ ]:
from pathlib import Path
import os, sys, zipfile, subprocess, json, hashlib
assert sys.version_info[:2] == (3, 11)
root = Path("handstand_v2_workspace")
assert not root.exists(), "Choose a new workspace; preserve previous results"
bundles = list(Path.cwd().glob("handstand_v2_molab*.zip"))
assert len(bundles) == 1, "Keep one uploaded Molab bundle beside this notebook"
with zipfile.ZipFile(bundles[0]) as z:
    z.extractall(root)
os.chdir(root)
manifest = json.loads(Path("handstand_v2_source_manifest.json").read_text())
for filename, expected in manifest["sha256"].items():
    assert hashlib.sha256(Path(filename).read_bytes()).hexdigest() == expected
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "handstand_v2_requirements.txt"], check=True)
os.environ["MUJOCO_GL"] = "egl"
os.environ["MPLCONFIGDIR"] = "/tmp/handstand_matplotlib"
def run(*args):
    subprocess.run([sys.executable, "-m", *args], check=True)


In [ ]:
run("pytest", "-q", "experiments/handstand_v2/test_measurement.py")
run("experiments.handstand_v2.run", "diagnose", "--output", "output/molab_v2_diagnostics")
run("experiments.handstand_v2.report", "output/molab_v2_diagnostics")

In [ ]:
run("experiments.handstand_v2.run", "smoke", "--output", "output/molab_v2_smoke")
print(Path("output/molab_v2_smoke/gate.json").read_text())

In [ ]:
gate = json.loads(Path("output/molab_v2_smoke/gate.json").read_text())
assert gate["pilot_go"] is True
run("experiments.handstand_v2.run", "pilot", "--gate", "output/molab_v2_smoke/gate.json", "--output", "output/molab_v2_pilot")
print(Path("output/molab_v2_pilot/gate.json").read_text())

Inspect both deterministic and stochastic matched-seed comparisons. Select the checkpoint with the best certified supported pitch while retaining takeoff. Do not select by raw peak alone. No extended run is launched by this notebook.

In [ ]:
run("experiments.handstand_v2.playback", "--output", "output/molab_v2_original.gif")
from IPython.display import display, Image, FileLink
display(Image(filename="output/molab_v2_original.gif"))
import shutil
shutil.make_archive("handstand_v2_complete", "zip", "output")
display(FileLink("handstand_v2_complete.zip"))